# 01 — Evaluación Comparativa: FoundationPose vs GDR-Net++

**TFM: Estimación de Pose 6-DoF mediante Transformers y Modelos de Difusión para Bin Picking Robótico**

Este notebook ejecuta inferencia y evaluación BOP de:
- **FoundationPose** (Wen et al., CVPR 2024) — Método principal
- **GDR-Net++** (Wang et al., CVPR 2021) — Baseline comparativo

Datasets: T-LESS, YCB-Video (BOP format)

⚠️ **Requiere GPU**: Ejecutar en Google Colab con runtime GPU (T4 gratuita)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/pose6dof-transformers-diffusion/blob/main/notebooks/01_gdrnet_foundationpose_eval.ipynb)

## 0. Setup del Entorno

In [ ]:
# Verificar GPU
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clonar nuestro repositorio TFM
!git clone https://github.com/Giocrisrai/pose6dof-transformers-diffusion.git tfm
%cd tfm
!pip install -q numpy scipy matplotlib opencv-python-headless trimesh open3d pycocotools

## 1. Descargar Datasets BOP (subset de test)

In [ ]:
import os
os.makedirs('data/datasets/tless', exist_ok=True)
os.makedirs('data/datasets/ycbv', exist_ok=True)

HF = 'https://huggingface.co/datasets/bop-benchmark'

# T-LESS
!wget -q -nc -P data/datasets/tless {HF}/tless/resolve/main/tless_base.zip
!wget -q -nc -P data/datasets/tless {HF}/tless/resolve/main/tless_models.zip
!wget -q -nc -P data/datasets/tless {HF}/tless/resolve/main/tless_test_primesense_all.zip

# YCB-Video
!wget -q -nc -P data/datasets/ycbv {HF}/ycbv/resolve/main/ycbv_base.zip
!wget -q -nc -P data/datasets/ycbv {HF}/ycbv/resolve/main/ycbv_models.zip
!wget -q -nc -P data/datasets/ycbv {HF}/ycbv/resolve/main/ycbv_test_all.zip

print('Downloads complete!')

In [ ]:
# Extraer
!cd data/datasets/tless && unzip -qo tless_base.zip && unzip -qo tless_models.zip && unzip -qo tless_test_primesense_all.zip
!cd data/datasets/ycbv && unzip -qo ycbv_base.zip && unzip -qo ycbv_models.zip && unzip -qo ycbv_test_all.zip

# Reorganizar estructura BOP
import shutil
for ds in ['tless', 'ycbv']:
    base = f'data/datasets/{ds}'
    inner = f'{base}/{ds}'
    if os.path.exists(inner):
        for f in os.listdir(inner):
            src = os.path.join(inner, f)
            dst = os.path.join(base, f)
            if not os.path.exists(dst):
                shutil.move(src, dst)

# Crear symlinks para models/
if not os.path.exists('data/datasets/tless/models'):
    os.symlink('models_cad', 'data/datasets/tless/models')

# camera.json
if os.path.exists('data/datasets/tless/camera_primesense.json'):
    shutil.copy('data/datasets/tless/camera_primesense.json', 'data/datasets/tless/camera.json')
if os.path.exists('data/datasets/ycbv/camera_uw.json'):
    shutil.copy('data/datasets/ycbv/camera_uw.json', 'data/datasets/ycbv/camera.json')

print('Datasets ready!')

In [ ]:
# Verificar datasets
from src.utils.dataset_loader import BOPDataset, verify_dataset

verify_dataset('data/datasets/tless', 'test')
verify_dataset('data/datasets/ycbv', 'test')

## 2. Setup GDR-Net++ (Baseline)

In [ ]:
# Clonar GDR-Net++
!git clone https://github.com/shanice-l/gdrnpp_bop2022.git /content/gdrnpp
%cd /content/gdrnpp

# Instalar dependencias
!pip install -q mmcv-full==1.7.0 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html
!pip install -q mmdet==2.28.2
!pip install -q -r requirements.txt
!python setup.py build_ext --inplace

print('GDR-Net++ installed!')

In [ ]:
# Descargar pesos pre-entrenados GDR-Net++
# NOTA: Actualizar estos links con los de OneDrive del repo oficial
# Password: groupji

# YOLOX detector
!mkdir -p pretrained_models/yolox
# !gdown <YOLOX_LINK> -O pretrained_models/yolox/

# GDR-Net++ trained models
!mkdir -p output/gdrn
# T-LESS: !gdown <TLESS_LINK> -O output/gdrn/tless/
# YCB-V: !gdown <YCBV_LINK> -O output/gdrn/ycbv/

print('Descargar pesos manualmente desde OneDrive (password: groupji)')
print('Links en: https://github.com/shanice-l/gdrnpp_bop2022#pretrained-models')

In [ ]:
# Ejecutar inferencia GDR-Net++ en T-LESS
# (Actualizar config path según los pesos descargados)

# !python core/gdrn_modeling/main_gdrn.py \
#     --config-file configs/gdrn/tless_pbr/convnext_a6_AugCosyAAEGray_BG05_mlL1_DMask_amodalClipBox_classAware_tless.py \
#     --eval-only \
#     --opts OUTPUT_DIR output/gdrn/tless \
#     DATASETS.TEST '("tless_bop_test",)'

print('TODO: Ejecutar después de descargar pesos')

## 3. Setup FoundationPose (Método Principal)

In [ ]:
# Clonar FoundationPose
%cd /content
!git clone https://github.com/NVlabs/FoundationPose.git
%cd FoundationPose

# Instalar dependencias
!pip install -q -r requirements.txt

# NVDiffRast (renderizado diferenciable)
!pip install -q git+https://github.com/NVlabs/nvdiffrast.git

# Build extensions
!bash build_all.sh

print('FoundationPose installed!')

In [ ]:
# Descargar pesos FoundationPose
!mkdir -p weights

# Los pesos oficiales están en el repo de FoundationPose
# Refiner: 2023-10-28-18-33-37
# Scorer: 2024-01-11-20-02-45

# NOTA: Verificar las instrucciones oficiales para la descarga
# https://github.com/NVlabs/FoundationPose#pretrained-models

print('Descargar pesos según instrucciones del repo oficial')

In [ ]:
# Inferencia FoundationPose en T-LESS
# (Actualizar paths)

# import sys
# sys.path.insert(0, '/content/FoundationPose')
# from estimater import FoundationPose, PoseRefinePredictor, ScorePredictor

# scorer = ScorePredictor()
# refiner = PoseRefinePredictor()
# model = FoundationPose(
#     model_pts=...,
#     model_normals=...,
#     scorer=scorer,
#     refiner=refiner,
# )

print('TODO: Ejecutar después de descargar pesos')

## 4. Evaluación Comparativa BOP

In [ ]:
import sys
sys.path.insert(0, '/content/tfm')

import json
import numpy as np
import matplotlib.pyplot as plt
from src.utils.metrics import add_metric, add_s_metric, compute_recall, compute_auc
from src.utils.visualization import plot_metrics_comparison

# Cargar resultados (después de ejecutar ambos métodos)
# fp_results = json.load(open('results/foundationpose_tless.json'))
# gdr_results = json.load(open('results/gdrnet_tless.json'))

# Ejemplo con datos simulados (reemplazar con resultados reales)
methods = ['GDR-Net++', 'FoundationPose']
vsd_scores = [62.3, 78.5]   # Recall VSD (literatura)
mssd_scores = [58.7, 74.2]  # Recall MSSD
mspd_scores = [65.1, 80.3]  # Recall MSPD

fig = plot_metrics_comparison(methods, vsd_scores, mssd_scores, mspd_scores,
                              title='BOP Metrics — T-LESS')
plt.savefig('experiments/results/comparison_tless.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en experiments/results/comparison_tless.png')

In [ ]:
# Tabla comparativa completa
from IPython.display import HTML

html = """
<table style='border-collapse:collapse; width:100%;'>
<tr style='background:#0098CD; color:white;'>
    <th style='padding:8px; border:1px solid #ddd;'>Método</th>
    <th style='padding:8px; border:1px solid #ddd;'>VSD ↑</th>
    <th style='padding:8px; border:1px solid #ddd;'>MSSD ↑</th>
    <th style='padding:8px; border:1px solid #ddd;'>MSPD ↑</th>
    <th style='padding:8px; border:1px solid #ddd;'>Score BOP ↑</th>
</tr>
<tr>
    <td style='padding:8px; border:1px solid #ddd;'>GDR-Net++ (baseline)</td>
    <td style='padding:8px; border:1px solid #ddd;'>62.3</td>
    <td style='padding:8px; border:1px solid #ddd;'>58.7</td>
    <td style='padding:8px; border:1px solid #ddd;'>65.1</td>
    <td style='padding:8px; border:1px solid #ddd;'>62.0</td>
</tr>
<tr style='background:#e6f7ff;'>
    <td style='padding:8px; border:1px solid #ddd;'><b>FoundationPose (ours)</b></td>
    <td style='padding:8px; border:1px solid #ddd;'><b>78.5</b></td>
    <td style='padding:8px; border:1px solid #ddd;'><b>74.2</b></td>
    <td style='padding:8px; border:1px solid #ddd;'><b>80.3</b></td>
    <td style='padding:8px; border:1px solid #ddd;'><b>77.7</b></td>
</tr>
</table>
<p><i>Nota: Valores de ejemplo basados en literatura. Reemplazar con resultados propios.</i></p>
"""
HTML(html)

## 5. Visualización de Resultados

In [ ]:
# Visualizar poses estimadas sobre imágenes de test
from src.utils.dataset_loader import BOPDataset
from src.utils.visualization import draw_pose_axes, draw_projected_points
import cv2

# Cargar un sample de T-LESS
ds = BOPDataset('data/datasets/tless', split='test')
scenes = ds.get_scene_ids()
if scenes:
    scene_id = scenes[0]
    sample = ds.load_sample(scene_id, 0)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(sample['rgb'])
    axes[0].set_title(f'RGB — Scene {scene_id}')
    axes[0].axis('off')
    
    # Depth (normalizado para visualización)
    depth_vis = sample['depth'].astype(float)
    depth_vis[depth_vis == 0] = np.nan
    axes[1].imshow(depth_vis, cmap='turbo')
    axes[1].set_title('Depth Map')
    axes[1].axis('off')
    
    plt.suptitle(f'T-LESS Test Sample — {len(sample["gt_poses"])} objects', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Info GT
    for i, gt in enumerate(sample['gt_poses']):
        print(f"  Object {i}: ID={gt['obj_id']}, t={gt['cam_t_m2c'][:2].round(1)} mm")
else:
    print('No test scenes found. Run dataset download cells first.')

In [ ]:
# Guardar resultados para usarlos localmente
# Los resultados se guardan en experiments/results/ y se pushean al repo

# !cp experiments/results/*.json /content/drive/MyDrive/TFM/results/
# !cp experiments/results/*.png /content/drive/MyDrive/TFM/results/

print('Resultados listos para descargar o copiar a Google Drive')

---

**Autores:** Giocrisrai Godoy · José Miguel Carrasco | **Directora:** Profesora Benítez | UNIR 2026